# HotpotQA dev: retrieval to RAG

Run the full HotpotQA validation split through the existing Contriever artifact, then generate answers with Llama-3.1-8B-Instruct without the compressor. Attach the artifact as a Kaggle Input and make sure your Hugging Face account can access Llama. Retrieval and generation run in separate processes so the retriever model is released before the reader loads.

In [ ]:
from pathlib import Path
import json, subprocess, sys

REPO = Path('/kaggle/working/EXIT')  # change if your checkout is elsewhere
ARTIFACT = Path('/kaggle/input/exit-retrieval-artifact')  # directory containing manifest.json
WORK = Path('/kaggle/working/hotpotqa-experiment')
QUESTIONS = WORK / 'hotpotqa-dev-questions.jsonl'
RETRIEVED = WORK / 'hotpotqa-dev-retrieved-top20.jsonl'
WORK.mkdir(parents=True, exist_ok=True)
assert REPO.is_dir(), f'Repository not found: {REPO}'
assert ARTIFACT.is_dir(), f'Artifact not found: {ARTIFACT}'
for name in ('manifest.json', 'corpus.sqlite', 'index.faiss', 'index_meta.faiss'):
    assert (ARTIFACT / name).is_file(), f'Missing artifact file: {name}'
print('GPU runtime and paths:', REPO, ARTIFACT, WORK)

## 1. Prepare the full validation split

Export id, question, and answers. The retrieval stage will fetch passages from your corpus and index. No row limit is applied.

In [ ]:
subprocess.run([sys.executable, str(REPO / 'prepare_reproduce_data.py'), '--output', str(QUESTIONS), '--questions-only', '--cache-dir', str(WORK / 'hf-cache')], cwd=REPO, check=True)
manifest = json.loads(QUESTIONS.with_suffix('.manifest.json').read_text(encoding='utf-8'))
assert manifest['exported_examples'] == manifest['source_split_examples']
assert manifest['skipped_example_count'] == 0
print('Exported validation examples:', manifest['exported_examples'])

## 2. Retrieval Top-20

The retriever uses `cuda:0` to encode questions; FAISS search runs on CPU. To resume, rerun the same command with the same output path.

In [ ]:
subprocess.run([sys.executable, '-m', 'retrieval.run', 'export', '--artifact', str(ARTIFACT), '--input', str(QUESTIONS), '--output', str(RETRIEVED), '--retriever-device', 'cuda:0', '--batch-size', '16', '--top-k', '20'], cwd=REPO, check=True)
retrieved_count = sum(1 for line in RETRIEVED.open(encoding='utf-8') if line.strip())
assert retrieved_count == manifest['exported_examples'], (retrieved_count, manifest['exported_examples'])
print('Retrieval export complete:', retrieved_count)

## 3. Validate retrieved records

Confirm every question has 20 passages before loading Llama.

In [ ]:
with RETRIEVED.open(encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        row = json.loads(line)
        assert row.get('question') and isinstance(row.get('answers'), list)
        assert isinstance(row.get('ctxs'), list) and len(row['ctxs']) >= 20, f'Row {i} has fewer than 20 passages'
print('JSONL validation passed')

## 4. Run the reader without compression

Rerun the same command to resume from saved progress. Leave --device unset so Transformers uses device_map=auto. Llama access must be enabled for your Hugging Face account.

In [ ]:
def run_reader(k):
    out = WORK / f'hotpotqa-dev-top{k}-no-compressor.jsonl'
    subprocess.run([sys.executable, str(REPO / 'reproduce.py'), '--input', str(RETRIEVED), '--output', str(out), '--dataset', 'hotpotqa', '--split', 'dev', '--top-k', str(k), '--reader-backend', 'transformers', '--reader-model', 'meta-llama/Llama-3.1-8B-Instruct', '--no-compressor', '--cache-dir', str(WORK / 'model-cache')], cwd=REPO, check=True)
    summary = out.with_suffix('.summary.json')

    return json.loads(summary.read_text(encoding='utf-8'))

summary_top5 = run_reader(5)
summary_top20 = run_reader(20)
print('Top-5 EM/F1:', summary_top5['metrics_percent'])
print('Top-20 EM/F1:', summary_top20['metrics_percent'])

## Results and resume

Keep WORK between Kaggle sessions to resume. Summaries contain EM/F1 and reader latency. Retrieval latency is recorded separately. This is the retrieval plus reader baseline without compression.